# 台湾电子产品外销订单：政府开放资料当前文件来源门

只核对政府当前 CSV 的身份、月度覆盖、元数据时钟和 Qlib 日历月份，不回填历史首次公开版本，不读取 SOXX 收益。

In [1]:
import os,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/taiwan_export_orders_open_data_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['budget']['model_fits']==plan['budget']['native_accounts']==0
for name,entry in plan['source_files'].items():assert sha(root/name)==entry['sha256'] and (root/name).stat().st_size==entry['size_bytes']
receipt=json.loads((card/'source_nas_archive.json').read_text());assert receipt['verified']
html=(root/'data/snapshots/taiwan_export_orders_open_data_20261023/dataset_16362.html').read_text();assert '2015-07-02' in html and '2026-09-21 13:37' in html
f=pd.read_csv(root/'data/snapshots/taiwan_export_orders_open_data_20261023/electronic_export_orders.csv',dtype=str)
assert list(f.columns)==['統計項目','貨品別','資料期(民國年)','統計值(金額)','計量單位']
assert f['統計項目'].eq('外銷訂單金額_美元').all() and f['貨品別'].eq('電子產品').all() and f['計量單位'].eq('百萬美元').all()
assert f['資料期(民國年)'].str.fullmatch(r'\d{5}').all();year=f['資料期(民國年)'].str[:3].astype(int)+1911;month=f['資料期(民國年)'].str[-2:].astype(int)
assert month.between(1,12).all();period=pd.PeriodIndex(year=year,month=month,freq='M');value=pd.to_numeric(f['統計值(金額)'],errors='raise').astype(float)
assert period.is_unique and period.is_monotonic_increasing and len(period)==len(pd.period_range(period.min(),period.max(),freq='M')) and np.isfinite(value).all() and (value>0).all()
source=pd.DataFrame({'observation_month':period.astype(str),'electronic_orders_million_USD':value,'original_ROC_month':f['資料期(民國年)']})
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/plan['calendar']['provider_uri']),region='us',kernels=1)
train_cal=pd.DatetimeIndex(D.calendar(start_time=plan['calendar']['train'][0],end_time=plan['calendar']['train'][1],freq='day'))
held_cal=pd.DatetimeIndex(D.calendar(start_time=plan['calendar']['reused_held'][0],end_time=plan['calendar']['reused_held'][1],freq='day'))
assert len(train_cal)==3020 and len(held_cal)==946
train_months=pd.PeriodIndex(train_cal,freq='M').unique();held_months=pd.PeriodIndex(held_cal,freq='M').unique();observed=set(period)
train_covered=sum(x in observed for x in train_months);held_covered=sum(x in observed for x in held_months)
audit={'source_rows':len(source),'first_observation_month':str(period.min()),'last_observation_month':str(period.max()),'single_product_statistic_and_unit':True,'consecutive_unique_months':True,'government_open_data_dataset_created':'2015-07-02','current_metadata_updated':'2026-09-21 13:37 Asia/Taipei as page display; source HTML exact bytes archived','original_monthly_release_vintages_qualified':False,'original_first_public_or_next_Stage1_decision_clock_qualified':False,'train_Qlib_signal_days':len(train_cal),'held_Qlib_signal_days':len(held_cal),'train_calendar_months':len(train_months),'train_months_in_current_file':train_covered,'held_calendar_months':len(held_months),'held_months_in_current_file':held_covered,'held_missing_current_file_months':[str(x) for x in held_months if x not in observed],'current_file_missing_published_August_2026_observation':str(period.max())<'2026-08','source_is_semiconductor_only':False,'SOXX_market_outcome_reads':0,'model_fits':0,'native_accounts':0}
tracking=card/'qlib_recorder';assert not tracking.exists()
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rid=rec.id;eid=str(rec.experiment_id);rec.save_objects(artifact_path='inputs',source_months=source,source_file_hashes=plan['source_files']);rec.save_objects(artifact_path='analysis',source_audit=audit,train_Qlib_calendar=train_cal,held_Qlib_calendar=held_cal);rec.log_metrics(source_months=len(source),train_Qlib_signal_days=len(train_cal),held_Qlib_signal_days=len(held_cal),model_fits=0,native_accounts=0)
summary={'experiment':plan['experiment'],'decision':'current_government_open_data_source_identity_and_coverage_passed_PIT_and_recent_vintage_missing','objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'source_nas_sha256':receipt['sha256'],'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'audit':audit,'artifact_policy':plan['artifact_policy'],'new_model_fits':0,'new_native_accounts':0,'next_action':'Find exact original monthly MOEA release bytes/clock and version history. The present government CSV is current revised history, created 2015 and stale at July2026 despite a released August2026 report; no Stage1 factor, information screen or 0bp account.'}
(card/'summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n');e=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='tw-orders-recorder-') as t:
 a=Path(t)/'recorder.tar.gz'
 with a.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(e['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(a)==e['sha256']
 with tarfile.open(a,'r:gz') as z:assert any(m.name.endswith('/artifacts/inputs/source_months') for m in z.getmembers())
summary['nas_evidence']=e;summary['nas_readback']={'verified':True,'sha256':e['sha256']};(card/'summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False)+'\n')
print({k:audit[k] for k in ('source_rows','first_observation_month','last_observation_month','train_months_in_current_file','held_months_in_current_file','held_missing_current_file_months')})


2026/10/23 06:52:22 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_10505/3179382568.py:16: FutureWarning: Constructing PeriodIndex from fields is deprecated. Use PeriodIndex.from_fields instead.
  assert month.between(1,12).all();period=pd.PeriodIndex(year=year,month=month,freq='M');value=pd.to_numeric(f['統計值(金額)'],errors='raise').astype(float)


[10505:MainThread](2026-10-23 06:52:23,433) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[10505:MainThread](2026-10-23 06:52:23,435) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


...

[10505:MainThread](2026-10-23 06:52:23,451) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name taiwan_export_orders_open_data_20261023.


[10505:MainThread](2026-10-23 06:52:23,453) INFO - qlib.workflow - [exp.py:258] - Experiment 957029813524552338 starts running ...


[10505:MainThread](2026-10-23 06:52:23,546) INFO - qlib.workflow - [recorder.py:345] - Recorder 77377a2fd33b4cc5a187a1e1985125b3 starts running under Experiment 957029813524552338 ...


[10505:MainThread](2026-10-23 06:52:24,686) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{'source_rows': 511, 'first_observation_month': '1984-01', 'last_observation_month': '2026-07', 'train_months_in_current_file': 144, 'held_months_in_current_file': 43, 'held_missing_current_file_months': ['2026-08', '2026-09', '2026-10']}


## 中文结论

# 台湾电子产品外销订单：政府开放资料当前文件来源门

台湾政府[开放资料集 16362](https://data.gov.tw/en/datasets/16362)标明发布者为经济部统计处、数据集发布日期 2015-07-02。经开放资料质量镜像取得的当前 CSV 有 511 个连续月份，1984-01 至 **2026-07**，字段均为“外销订单金额（美元）／电子产品／百万美元”。这是广义电子产品接单额，不是半导体专属订单。原始 MOEA 页与统计服务本次各返回 HTTP 403；四次 GET 中两次 403、两次政府镜像 200，均已记账。冻结计划中的“2 次 GET”只覆盖成功文件，不能当作本轮总请求数。

用冻结原 Stage1 Qlib 日历只核对观察月份覆盖：2011–2022 训练 3,020 信号日对应的 144 个月都出现在**当前历史文件**；2023–2026-10-09 复用段 946 信号日涉及 46 个月，仅 43 个月在该文件，2026-08/09/10 不在内。其中 2026-10 尚是未完成观察月，不能与已经公布但镜像缺失的月份混为一谈。政府页面显示当前资料更新时间 2026-09-21 13:37，而 CSV 最后观察月仍为 2026-07；[经济部 2026 年 8 月订单发布页](https://www.moea.gov.tw/Mns/populace/news/EpaperDetail.aspx?volume=20260922)的官方搜索索引显示 9 月发布线索，但原页仍不可直读，因此历史与最近月份的原始发布版本、修订及下一次美股 Stage1 决策前可用时刻均**未合格**。2015 年前的数值在当前数据集内也只是回填历史，不能作为当时已获得本数据集的证据。

原始政府镜像 HTML、CSV、响应元数据 NAS 归档 SHA-256 `5f9634a92ed21879f38f2d8fec8b9c171e94a010831f57f10af96b25427185aa`；两份 403 原响应与四次 GET 账本补充归档 `8897297d6e1fdc95363159a21fdf364777237ab1d10d927dae3e1098150dddaa`；原生 Qlib 来源审计 Recorder `77377a2fd33b4cc5a187a1e1985125b3` 的 NAS 归档 `860e65f5930f60e9b6174f76c0f4e101147e9817cea9437e0ecdbb794c4c0e95`。三组均经远端哈希与限定恢复。独立复核确认**当前数值身份**，没有批准历史 PIT 因子。0 SOXX 结果读取、0 拟合、0 账户。

决定 `current_government_mirror_numeric_identity_only_PIT_and_recent_months_missing`。与已失败的工业产量、韩国实际出口和单一公司指引相比，接单需求在机制上不同；但缺原始逐月发布和最近观察，不能把当前修订 CSV 映射成历史 Stage1 信号，也不能由本卡声称费用后超过 SOXX。
